![ClimaSense](https://raw.githubusercontent.com/Ramesh-Arvind/climasense-/main/docs/thumbnail.png)

# ClimaSense on Google Colab

**Gemma 4 Good Hackathon — Global Resilience Track**

This is a one-click runnable demo. Steps:
1. **Runtime → Change runtime type → T4 GPU**
2. Add your Hugging Face token as a Colab secret named `HF_TOKEN` (left sidebar → 🔑). Get one at https://huggingface.co/settings/tokens. Make sure you've accepted the Gemma 4 license at https://huggingface.co/google/gemma-4-E4B-it.
3. **Runtime → Run all**.

First run takes ~3 min (model download). Subsequent cells run in seconds.

In [ ]:
import os, sys
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# Install ClimaSense from GitHub
!pip install -q --upgrade pip
!pip install -q 'transformers>=5.5.0' --upgrade 'bitsandbytes>=0.45.0' 'accelerate>=0.30.0' deep-translator gtts
!pip install -q git+https://github.com/Ramesh-Arvind/climasense-.git

# Force-reload in case Colab preloaded an older transformers
for m in list(sys.modules):
    if m.startswith('transformers'):
        del sys.modules[m]
import transformers, torch
print('transformers:', transformers.__version__)
print('torch CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Hugging Face auth (Gemma 4 is gated)
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [ ]:
from climasense.agent import ClimaSenseAgent

agent = ClimaSenseAgent(
    model_id='google/gemma-4-E4B-it',  # E4B fits T4 in 4-bit
    audio_model_id=None,
    max_turns=4,
)
agent.load_model()
print('Agent ready.')

## Scenario 1 — English text query (Kenyan smallholder)

In [ ]:
result = agent.run(
    query='My maize leaves are turning yellow at the edges. Should I plant tomorrow? Rain coming?',
    location=(-0.0917, 34.7680),  # Kisumu, Kenya
)
print('Tools used:', [t['tool'] for t in result.get('tool_calls', [])])
print()
print(result['response'])

## Scenario 2 — Vision: diagnose a leaf photo

In [ ]:
import requests, io
from PIL import Image

LEAF_URL = 'https://raw.githubusercontent.com/Ramesh-Arvind/climasense-/main/data/demo_crops/early_blight_alternaria_solani_01.jpg'
leaf = Image.open(io.BytesIO(requests.get(LEAF_URL).content)).convert('RGB')
display(leaf)

result_vis = agent.run(
    query='Look at this leaf. What is wrong with my crop and what should I do?',
    images=[leaf],
    location=(-0.0917, 34.7680),
)
print()
print(result_vis['response'])

## Scenario 3 — Swahili query (auto language matching)

In [ ]:
result_sw = agent.run(
    query='Tafadhali nisaidie. Mahindi yangu yana matatizo na ninataka kujua kuhusu hali ya hewa.',
    location=(-6.1722, 35.7395),  # Dodoma, Tanzania
)
print(result_sw['response'])

# Optional English translation for non-Swahili reviewers
try:
    from deep_translator import GoogleTranslator
    print('\n--- English (Google Translate) ---')
    print(GoogleTranslator(source='sw', target='en').translate(result_sw['response']))
except Exception as e:
    print('translation skipped:', e)

---
**Source:** https://github.com/Ramesh-Arvind/climasense-

**Architecture:** ![arch](https://raw.githubusercontent.com/Ramesh-Arvind/climasense-/main/docs/architecture.png)